# PICKO Research · NB1 — **Breadth**: how many tools before selection breaks?

We finetune **one** model on the 40 focus tools and then, on its **held-out test set**, offer each query a
random subset of `k` tools (its gold tool + `k-1` distractors) and measure tool-selection accuracy. Training
is fixed, so the curve isolates a single variable — *how many tools are offered at inference* — averaged
over `N_REPEATS` random subsets per `k` (mean ± std). The 1024-token encoder truncates long compact lists,
so past ~20 tools some are never seen; that ceiling is the result, annotated with `n_visible`.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones **upstream Needle** (Cactus, pinned commit) and installs it, clones this **PICKO** repo,
pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and the checkpoints+results (out)
at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_training_pool.jsonl` must be in `MyDrive/picko/`. **Running locally?**
This cell is a no-op — install Needle yourself (`pip install -e /path/to/needle`) and skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
# Upstream Needle (Cactus) — de-vendored: cloned + installed at a pinned commit.
NEEDLE_REPO = "https://github.com/cactus-compute/needle.git"
NEEDLE_SHA  = "34861f39ae292429f80a62c96abe83218a852d57"   # pinned; has _per_tool_split — update if upstream drifts
# This PICKO repo — the scripts/notebooks/data imported below.
PICKO_REPO   = "https://github.com/HadarBit/picko.git"     # <- set to your submission repo URL
PICKO_BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("/content/needle"):
        !git clone -q {NEEDLE_REPO} /content/needle && cd /content/needle && git checkout -q {NEEDLE_SHA}
    if not os.path.exists("/content/picko"):
        !git clone -q -b {PICKO_BRANCH} {PICKO_REPO} /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    %pip install -q -e /content/needle                          # install upstream Needle
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_training_pool.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_training_pool.jsonl", "/content/drive/MyDrive/picko_training_pool.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_training_pool.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    try:                                                        # guard: upstream must expose the split PICKO uses
        from needle.training.finetune import _per_tool_split    # noqa: F401
    except Exception as e:
        raise ImportError(f"Upstream Needle @ {NEEDLE_SHA[:7]} lacks _per_tool_split ({e}). "
                          "Pin NEEDLE_SHA to a commit that has it, or re-run the install.")
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip lines above, then restart.")
    print(f"bootstrap OK · GPU active · needle@{NEEDLE_SHA[:7]} · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally. Install upstream Needle first: pip install -e /path/to/needle")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Train / test split

Both the training subprocess and this notebook call the **same deterministic** `per_tool_split`
(`seed=42`, 10 test + 10 val per tool). The model trains **only on the train split**; the sweep below runs
**only on the held-out test split**, so no test query is ever seen in training.

## 3 · Configure

In [ ]:
NB_DIR = os.path.join(OUT_DIR, "nb1"); os.makedirs(NB_DIR, exist_ok=True)   # this notebook's outputs
BREADTH_SIZES  = [3, 5, 10, 20, 30, 40]   # tools offered per query at inference
N_REPEATS      = 8                        # random subsets averaged per size (mean +/- std)
CAP_PER_TOOL   = 120                       # examples/tool -> 100 train / 10 val / 10 test
EPOCHS         = 1
EVAL_SUBSAMPLE = 100                       # test queries per (size, repeat), sampled across all tools; None = full
MAX_GEN_LEN    = 64
BATCH_SIZE     = 8                         # lower to 4 on OOM, raise to 16 if headroom
RUN_TRAIN      = True
FORCE_RETRAIN  = False
print("focus tools:", len(FOCUS), "| sizes:", BREADTH_SIZES, "| repeats/size:", N_REPEATS, "| out:", NB_DIR)

## 4 · Train the model (40 focus tools, compact)\nTrained once to Drive and reused; the returned `test` set is the held-out split used by the sweep.

In [ ]:
FOCUS40 = finetune_and_eval(cat, raw, tok, FOCUS, "breadth_focus40", NB_DIR,
                            cap=CAP_PER_TOOL, epochs=EPOCHS, compact=True, offer_all=len(FOCUS),
                            eval_subsample=None, run_train=RUN_TRAIN, force_retrain=FORCE_RETRAIN,
                            max_gen_len=MAX_GEN_LEN, batch_size=BATCH_SIZE)
m40, p40, tk40 = FOCUS40["bundle"]
TEST = FOCUS40["test"]                     # held-out test queries (never trained on)
log(f"model ready · held-out test queries={len(TEST)}")

## 5 · Sweep: offer k tools per query, repeat, average\nInference only, on the held-out test set. Resumable: finished `(k, repeat)` pairs persist to `nb1/breadth_results.json`.

In [ ]:
import contextlib, io
RES = os.path.join(NB_DIR, "breadth_results.json")
rows = json.load(open(RES)) if (os.path.exists(RES) and not FORCE_RETRAIN) else []
done = {(r["k"], r["repeat"]) for r in rows}
if done: log(f"resumed {len(done)} finished (k,repeat) run(s)")

def gold_of(ex):
    calls = json.loads(ex.get("answers", "[]"))
    return next((c["name"] for c in calls if isinstance(c, dict) and c.get("name")), None)

t_all = time.time()
for k in BREADTH_SIZES:
    for rep in range(N_REPEATS):
        if (k, rep) in done and not FORCE_RETRAIN: continue
        try:
            test = TEST[:EVAL_SUBSAMPLE] if EVAL_SUBSAMPLE else TEST
            offered = [offer_subset(cat, gold_of(e), FOCUS, k, seed=k*100000+rep*1000+i)
                       for i, e in enumerate(test)]
            with contextlib.redirect_stdout(io.StringIO()):
                preds = predict(m40, p40, tk40, test, tools_override=offered, max_gen_len=MAX_GEN_LEN)
            met = evaluate(test, preds, family_of=family_of)
            vis = int(np.median([n_visible(e["query"], json.loads(o), tok) for e, o in zip(test, offered)]))
            rows = [r for r in rows if not (r["k"] == k and r["repeat"] == rep)] + [{
                "k": k, "repeat": rep, "selection_acc": met["selection_acc"],
                "name_f1": met["name_f1"], "parse_rate": met["parse_rate"],
                "n_visible": vis, "n_test": met["n"]}]
            done.add((k, rep)); json.dump(rows, open(RES, "w"), indent=2)
            log(f"k={k} rep={rep}: selection={met['selection_acc']:.3f} visible={vis}/{k}")
        except Exception as e:
            log(f"k={k} rep={rep}: FAILED ({type(e).__name__}: {e})")

log(f"ALL RUNS DONE in {time.time()-t_all:.0f}s · results={RES}")
runs = pd.DataFrame(rows)
breadth = (runs.groupby("k")
           .agg(selection_mean=("selection_acc", "mean"), selection_std=("selection_acc", "std"),
                name_f1_mean=("name_f1", "mean"), parse_rate=("parse_rate", "mean"),
                n_visible=("n_visible", "median"), n_repeats=("repeat", "nunique"),
                n_test=("n_test", "sum")).reset_index())
breadth["selection_std"] = breadth["selection_std"].fillna(0)
display(breadth.round(3))

## 6 · The Breadth curve

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.errorbar(breadth["k"], breadth["selection_mean"], yerr=breadth["selection_std"],
            fmt="o-", color="#4C72B0", capsize=4, label="selection_acc (mean +/- std)")
ax.scatter(runs["k"], runs["selection_acc"], s=12, color="#4C72B0", alpha=0.25, zorder=1)
wall = breadth[breadth["n_visible"] < breadth["k"]]
if len(wall):
    kw = int(wall["k"].iloc[0]); vw = int(wall["n_visible"].iloc[0])
    ax.axvline(kw, color="#C44E52", ls=":", lw=1.5)
    ax.text(kw, 0.06, f" truncation wall\n (~{vw} of {kw} tools visible)", color="#C44E52", fontsize=9, va="bottom")
ax.set_xlabel("# tools offered at inference (k)"); ax.set_ylabel("tool-selection accuracy")
ax.set_ylim(0,1.02); ax.set_title(f"Breadth: selection vs #tools offered ({int(breadth['n_repeats'].max())} subsets/size)")
ax.legend()
plt.tight_layout(); save_fig("breadth_curve", out_dir=NB_DIR); plt.show()

## 7 · Read-out

One model, probed on held-out queries with random subsets of `k` tools, so the curve reflects the *offered*
tool count alone — the error bars show the spread a single sample per size would hide. Selection stays high
for small `k` and falls as `k` grows past the red line, where the compact offered list overflows the
1024-token encoder and the extra tools are truncated away. **Takeaway:** one PICKO instance is bounded by
the context window it can offer, not by what it was trained on — beyond the wall, shard the tool set.